# 08 深層学習モデル(Tier 3 — cross-sectional MLP)

Tier 3 は scikit-learn の `MLPRegressor`(フィードフォワード NN)を **Tier 0-2 と同じ (date, asset) 設計行列・同じ fit/predict 契約・同じ walk-forward** に乗せる(torch 不要)。非線形モデルは **同じ OOS walk で** ベースライン/線形/木を超えて初めて価値がある。

In [ ]:
import numpy as np
import pandas as pd

# Synthetic panel where forward returns are partly predictable (mild AR(1)
# momentum) so a model has signal to learn. No network.
rng = np.random.default_rng(3)
idx = pd.bdate_range('2015-01-01', periods=1600)
names = [f'A{i}' for i in range(8)]
n, k = len(idx), len(names)
r = rng.normal(0, 0.012, (n, k))
for t in range(1, n):
    r[t] += 0.06 * r[t - 1]
rets = pd.DataFrame(r, index=idx, columns=names)
prices = 100 * np.exp(rets.cumsum())
prices.iloc[-1].round(1)

In [ ]:
from quantkit import features as F, models as MD, signals as S, backtest as B
from quantkit.labels import forward_return
from quantkit.features.price import cross_sectional_zscore
from quantkit.utils.config import load_config

H = 5
feats = {
    'mom_120_20': F.momentum(prices, lookback=120, skip=20),
    'z_20': F.rolling_zscore(prices, 20),
    'rvol_20': F.realized_volatility(prices, 20),
    'ma_ratio_50': F.ma_ratio(prices, 50),
}
X, y = MD.make_design(feats, forward_return(prices, H))
folds = B.walk_forward(prices.index, train=252, test=63, horizon=H, embargo=3)
print(f'{len(folds)} folds, X={X.shape}')

## OOS 予測: Tier0 mean / Tier1 ridge / Tier2 RF / **Tier3 MLP**

In [ ]:
factories = {
    'mean': lambda: MD.MeanModel(),
    'ridge': lambda: MD.ridge(alpha=1.0),
    'random_forest': lambda: MD.random_forest(n_estimators=150, max_depth=3),
    'mlp': lambda: MD.mlp(hidden_layer_sizes=(32, 16), max_iter=300),
}
oos = {m: MD.walk_forward_predict(f, X, y, folds) for m, f in factories.items()}
print('OOS predictions:', {m: len(p) for m, p in oos.items()})

## weights(horizon 一致リバランス)→ 同一エンジンで比較

In [ ]:
cost = B.CostModel.from_config(load_config('backtest_config'))
oos_dates = pd.DatetimeIndex(sorted(set().union(*[p.index.get_level_values('date') for p in oos.values()])))
results = {}
for m, pred in oos.items():
    panel = MD.predictions_to_panel(pred).reindex(oos_dates)
    w = S.long_short_quantile(cross_sectional_zscore(panel).iloc[::H], 0.3).reindex(oos_dates, method='ffill')
    results[m] = B.run_backtest(w, rets, cost_model=cost, lag=1)
results['equal_weight'] = B.buy_and_hold(rets.loc[oos_dates])
B.compare(results, periods=252).round(4)

**読み方**: MLP(Tier3)の Sharpe が Tier0-2 と等加重を OOS で超えるか。超えなければ「この問題に深層は過剰」を提示する(それも結果)。

**正直な注記**: 合成データ・少資産・単純コスト・ハイパラ未調整。MLP は小規模・乱数依存。*方法論*の実演で投資結果ではない。Sequence(LSTM/TCN)や基盤モデルは Tier4(NB09)。